In [ ]:
!pip install -q -U transformers datasets sentence-transformers pandas faiss-cpu peft accelerate torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 136.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1

In [ ]:
import torch
import time
import pandas as pd
import numpy as np
import faiss
import pickle
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer
from sentence_transformers import SentenceTransformer
from peft import PeftModel
from tqdm.auto import tqdm
from google.colab import userdata, drive
import os

# =====================================================================
# 1. THE BOOT SEQUENCE
# =====================================================================
print("🚀 BOOTING THE ARENA (Loading all models onto A100)...")
drive.mount('/content/drive')
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
ROUTER_DIR = "/content/drive/MyDrive/Adaptive_RAG_Project/llama3-medhallu-router"

print("--> Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

print("--> Loading Base Generator (Causal LM)...")
# Used for generating the actual answers
gen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map="cuda:0", torch_dtype=torch.bfloat16
)

print("--> Loading LoRA Router (Sequence Classification)...")
# Used for predicting hallucinations before they happen
base_router = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=2, device_map="cuda:0", torch_dtype=torch.bfloat16
)
router_model = PeftModel.from_pretrained(base_router, ROUTER_DIR)
router_model.eval()

print("--> Loading RAG Backend (FAISS & Embedder)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2', device="cuda:0")

# Loading your existing FAISS database!
rag_index = faiss.read_index("/content/drive/MyDrive/Adaptive_RAG_Project/FAISS/pubmed_massive_faiss.index")
with open("/content/drive/MyDrive/Adaptive_RAG_Project/FAISS/pubmed_massive_mapping.pkl", "rb") as f:
    rag_mapping = pickle.load(f)

print("✅ ALL SYSTEMS GO.\n")

# =====================================================================
# 2. THE THREE SYSTEMS (Core Functions)
# =====================================================================



# =====================================================================
# THE CORRECTIONS:
# 1. do_sample=False added to all generation to fix the impossible math.
# 2. System A stripped of context so it actually runs fast.
# 3. Router given a threshold to bypass RAG more often.
# =====================================================================

def system_a_parametric(question):
    """Answers strictly from internal memory. NO CONTEXT PROVIDED."""
    start_time = time.time()
    prompt = f"Question: {question}\n\nProvide a concise medical answer:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(gen_model.device)

    # NEW: Track input tokens
    input_tokens = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = gen_model.generate(**inputs, max_new_tokens=50, pad_token_id=tokenizer.eos_token_id, do_sample=False)

    answer = tokenizer.decode(outputs[0][input_tokens:], skip_special_tokens=True).strip()
    return answer, time.time() - start_time, input_tokens

def system_b_rag(question, k=5):
    """Retrieves top-k FAISS abstracts and forces LLM to read all of them."""
    start_time = time.time()

    query_emb = np.array([embedder.encode(question)]).astype('float32')
    faiss.normalize_L2(query_emb)

    # NEW: Retrieve 5 abstracts instead of 1
    distances, indices = rag_index.search(query_emb, k=k)
    retrieved_text = "\n\n".join([rag_mapping['texts'][idx] for idx in indices[0]])

    prompt = f"Context: {retrieved_text}\n\nQuestion: {question}\n\nBased strictly on the context, provide a concise medical answer:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(gen_model.device)

    # NEW: Track input tokens
    input_tokens = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = gen_model.generate(**inputs, max_new_tokens=50, pad_token_id=tokenizer.eos_token_id, do_sample=False)

    answer = tokenizer.decode(outputs[0][input_tokens:], skip_special_tokens=True).strip()
    return answer, time.time() - start_time, input_tokens, retrieved_text

def lora_router(question):
    """Predicts hallucination using softmax confidence threshold."""
    start_time = time.time()
    prompt = f"Context: \n\nQuestion: {question}\n\nProvide a concise medical answer:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(router_model.device)

    # NEW: Track router tokens
    router_tokens = inputs['input_ids'].shape[1]

    with torch.no_grad():
        logits = router_model(**inputs).logits
        probs = torch.nn.functional.softmax(logits, dim=-1)
        hallucination_risk = probs[0][1].item()
        prediction = 1 if hallucination_risk > 0.70 else 0

    return prediction, time.time() - start_time, router_tokens

# =====================================================================
# THE SHOOTOUT LOOP
# =====================================================================

print("📊 INITIALIZING THE SHOOTOUT...")
dataset = load_dataset("pubmed_qa", "pqa_labeled", split="train")
df_eval = pd.DataFrame(dataset)
df_test = df_eval.sample(n=100, random_state=42).reset_index(drop=True)

metrics = {
    "A_Parametric": {"correct": 0, "total_time": 0.0, "total_tokens": 0},
    "B_RAG": {"correct": 0, "total_time": 0.0, "total_tokens": 0},
    "C_Adaptive": {"correct": 0, "total_time": 0.0, "total_tokens": 0, "rag_bypassed": 0}
}

print(f"Executing Benchmark on {len(df_test)} queries...\n")

for idx, row in tqdm(df_test.iterrows(), total=len(df_test)):
    question = row['question']
    ground_truth = row['final_decision'].strip().lower()

    # System A
    ans_a, time_a, tokens_a = system_a_parametric(question)
    metrics["A_Parametric"]["total_time"] += time_a
    metrics["A_Parametric"]["total_tokens"] += tokens_a
    if ground_truth in ans_a.lower(): metrics["A_Parametric"]["correct"] += 1

    # System B
    ans_b, time_b, tokens_b, _ = system_b_rag(question)
    metrics["B_RAG"]["total_time"] += time_b
    metrics["B_RAG"]["total_tokens"] += tokens_b
    if ground_truth in ans_b.lower(): metrics["B_RAG"]["correct"] += 1

    # System C (Router Tokens + Execution Tokens)
    route_decision, time_route, tokens_route = lora_router(question)
    metrics["C_Adaptive"]["total_time"] += time_route
    metrics["C_Adaptive"]["total_tokens"] += tokens_route

    if route_decision == 0:
        ans_c, time_c, tokens_c = system_a_parametric(question)
        metrics["C_Adaptive"]["rag_bypassed"] += 1
    else:
        ans_c, time_c, tokens_c, _ = system_b_rag(question)

    metrics["C_Adaptive"]["total_time"] += time_c
    metrics["C_Adaptive"]["total_tokens"] += tokens_c
    if ground_truth in ans_c.lower(): metrics["C_Adaptive"]["correct"] += 1

# =====================================================================
# FINAL RESULTS DASHBOARD
# =====================================================================
total = len(df_test)
print("\n" + "="*60)
print(" 🏆 FINAL ARCHITECTURE BENCHMARK RESULTS")
print("="*60)
print(f"Total Test Queries: {total}\n")

print("1️⃣ SYSTEM A (Vanilla LLM - Always Parametric)")
print(f"   Accuracy:        {(metrics['A_Parametric']['correct'] / total) * 100:.1f}%")
print(f"   Latency:         {metrics['A_Parametric']['total_time']:.2f} seconds")
print(f"   Tokens Read:     {metrics['A_Parametric']['total_tokens']:,}")

print("\n2️⃣ SYSTEM B (Standard Setup - Always RAG)")
print(f"   Accuracy:        {(metrics['B_RAG']['correct'] / total) * 100:.1f}%")
print(f"   Latency:         {metrics['B_RAG']['total_time']:.2f} seconds")
print(f"   Tokens Read:     {metrics['B_RAG']['total_tokens']:,}")

print("\n3️⃣ SYSTEM C (Our Architecture - Adaptive RAG via LoRA)")
print(f"   Accuracy:        {(metrics['C_Adaptive']['correct'] / total) * 100:.1f}%")
print(f"   Latency:         {metrics['C_Adaptive']['total_time']:.2f} seconds")
print(f"   Tokens Read:     {metrics['C_Adaptive']['total_tokens']:,}")
print(f"   Compute Savings: Bypassed RAG {metrics['C_Adaptive']['rag_bypassed']}% of the time.")

# Calculate the financial/compute savings
saved_tokens = metrics['B_RAG']['total_tokens'] - metrics['C_Adaptive']['total_tokens']
print(f"   Efficiency:      Saved {saved_tokens:,} input tokens compared to Standard RAG!")
print("="*60)

🚀 BOOTING THE ARENA (Loading all models onto A100)...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--> Loading Tokenizer...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Executing Benchmark on 100 queries...



  0%|          | 0/100 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!



 🏆 FINAL ARCHITECTURE BENCHMARK RESULTS
Total Test Queries: 100

1️⃣ SYSTEM A (Vanilla LLM - Always Parametric)
   Accuracy:        53.0%
   Latency:         187.64 seconds
   Tokens Read:     2,758

2️⃣ SYSTEM B (Standard Setup - Always RAG)
   Accuracy:        59.0%
   Latency:         201.02 seconds
   Tokens Read:     151,665

3️⃣ SYSTEM C (Our Architecture - Adaptive RAG via LoRA)
   Accuracy:        58.0%
   Latency:         202.71 seconds
   Tokens Read:     123,584
   Compute Savings: Bypassed RAG 22% of the time.
   Efficiency:      Saved 28,081 input tokens compared to Standard RAG!
